In [ ]:
from data_pipeline import load_and_prepare, engineer_features, stratified_group_split

# 7 raw WLT parquet files (DPK456_11..17) — same source as map_WLT_value_to_CQ_slippage
_BASE = r"G:\DfsDE\LOC\Rt\BST\09_projects\BMI420\External\02_Product_Development\03_System\04_Accel_System_Development\10_ASIC_evaluations\CA\lot1_lot2_lot3_wafermap_tp_V2"
PARQUET_FILES = [
    rf"{_BASE}\DPK456_11.parquet",
    rf"{_BASE}\DPK456_12.parquet",
    rf"{_BASE}\DPK456_13.parquet",
    rf"{_BASE}\DPK456_14.parquet",
    rf"{_BASE}\DPK456_15.parquet",
    rf"{_BASE}\DPK456_16.parquet",
    rf"{_BASE}\DPK456_17.parquet",
]

TARGET = 'WLT_fail'

pivoted = load_and_prepare(PARQUET_FILES)
data, features_clean = engineer_features(pivoted, target=TARGET)
split = stratified_group_split(data, target=TARGET)

Feature reduction: 50 -> 19 (0 near-constant, 31 correlated)

Per-wafer fail rates:
  DPK456-11-C0 (7366 dies): hot_part_fail: 0.042 | cold_part_fail: 0.044
  DPK456-12-E3 (7372 dies): hot_part_fail: 0.032 | cold_part_fail: 0.036
  DPK456-13-G6 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.054
  DPK456-14-B6 (7356 dies): hot_part_fail: 0.033 | cold_part_fail: 0.038
  DPK456-15-E1 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.053
  DPK456-16-G4 (7384 dies): hot_part_fail: 0.034 | cold_part_fail: 0.037
  DPK456-17-B4 (7367 dies): hot_part_fail: 0.039 | cold_part_fail: 0.042

Stratified split:
  Train:   22101 dies / 3 wafers ['DPK456-13-G6', 'DPK456-14-B6', 'DPK456-16-G4']
  Val:     14733 dies / 2 wafers ['DPK456-11-C0', 'DPK456-17-B4']
  Holdout: 14733 dies / 2 wafers ['DPK456-12-E3', 'DPK456-15-E1']
  hot_part_fail fail rate — Train: 0.0390 | Val: 0.0400 | Holdout: 0.0411
  cold_part_fail fail rate — Train: 0.0426 | Val: 0.0432 | Holdout: 0.0445


# PREDICTION

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                              f1_score, precision_recall_curve, auc)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import json
import os
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Business cost parameters ---
# Cost of missing a fail (FN) vs falsely flagging a pass (FP)
FN_COST = 10
FP_COST = 1
RECALL_AT_PRECISION_TARGET = 0.8
N_OPTUNA_TRIALS = 50
OPTUNA_PARAMS_FILE = 'optuna_best_params.json'

# --- Unpack split ---
train_idx = split['train_idx']
val_idx = split['val_idx']
holdout_idx = split['holdout_idx']

X = data[features_clean].values

# --- Encode target: WLT_fail (CP2/CP3/CP4/Pass) ---
le = LabelEncoder()
y_all = le.fit_transform(data[TARGET].values)
class_names = list(le.classes_)
n_classes = len(class_names)
print(f"Target: {TARGET}")
print(f"Classes: {class_names} (encoded 0..{n_classes-1})")
print(f"Class distribution: {dict(zip(class_names, np.bincount(y_all)))}")

X_train, y_train = X[train_idx], y_all[train_idx]
X_val, y_val = X[val_idx], y_all[val_idx]
X_holdout, y_holdout = X[holdout_idx], y_all[holdout_idx]
sample_weights = compute_sample_weight('balanced', y_train)

# --- Optuna HPO for multi-class XGBoost ---
def make_objective(X_tr, y_tr, X_v, y_v, sw):
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        m = XGBClassifier(**params, random_state=42, eval_metric='mlogloss',
                          early_stopping_rounds=20, objective='multi:softprob',
                          num_class=n_classes)
        m.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_v, y_v)], verbose=False)
        y_pred = m.predict(X_v)
        # Minimize: weighted cost of misclassifying fails as pass + pass as fail
        cm_v = confusion_matrix(y_v, y_pred, labels=list(range(n_classes)))
        pass_idx = list(le.classes_).index('Pass')
        # FN = fails predicted as Pass; FP = Pass predicted as fail
        fn = cm_v[:, pass_idx].sum() - cm_v[pass_idx, pass_idx]  # fail rows predicted Pass
        fp = cm_v[pass_idx, :].sum() - cm_v[pass_idx, pass_idx]  # Pass rows predicted fail
        return FN_COST * fn + FP_COST * fp
    return objective

# Load or run HPO
saved_params = {}
if os.path.exists(OPTUNA_PARAMS_FILE):
    with open(OPTUNA_PARAMS_FILE, 'r') as f:
        saved_params = json.load(f)
    print(f"\nLoaded saved Optuna params from {OPTUNA_PARAMS_FILE}")

if TARGET in saved_params:
    bp = saved_params[TARGET]
    print(f"Using saved params: {bp}")
else:
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(X_train, y_train, X_val, y_val, sample_weights),
                   n_trials=N_OPTUNA_TRIALS)
    bp = study.best_params
    print(f"Optuna best val cost: {study.best_value}")
    print(f"Best params: {bp}")

# Save params
saved_params[TARGET] = bp
with open(OPTUNA_PARAMS_FILE, 'w') as f:
    json.dump(saved_params, f, indent=2)

# --- Train final model ---
model = XGBClassifier(**bp, random_state=42, eval_metric='mlogloss',
                      early_stopping_rounds=20, objective='multi:softprob',
                      num_class=n_classes)
model.fit(X_train, y_train, sample_weight=sample_weights,
          eval_set=[(X_val, y_val)], verbose=False)
print(f"Early stopping: iteration {model.best_iteration} / {bp['n_estimators']}")

# --- Holdout evaluation ---
y_hold_pred = model.predict(X_holdout)
y_hold_proba = model.predict_proba(X_holdout)

acc = accuracy_score(y_holdout, y_hold_pred)
f1_macro = f1_score(y_holdout, y_hold_pred, average='macro')
f1_weighted = f1_score(y_holdout, y_hold_pred, average='weighted')
cm = confusion_matrix(y_holdout, y_hold_pred, labels=list(range(n_classes)))
report = classification_report(y_holdout, y_hold_pred, labels=list(range(n_classes)),
                               target_names=class_names, output_dict=True)

# Business cost on holdout
pass_idx = class_names.index('Pass')
fn = cm[:, pass_idx].sum() - cm[pass_idx, pass_idx]
fp = cm[pass_idx, :].sum() - cm[pass_idx, pass_idx]
holdout_cost = int(FN_COST * fn + FP_COST * fp)

# Per-class PR-AUC (one-vs-rest for each fail class)
pr_aucs = {}
for i, cls in enumerate(class_names):
    if cls == 'Pass':
        continue
    y_bin = (y_holdout == i).astype(int)
    proba_i = y_hold_proba[:, i]
    prec, rec, _ = precision_recall_curve(y_bin, proba_i)
    pr_aucs[cls] = auc(rec, prec)

print(f"\nHoldout Results:")
print(f"  Accuracy: {acc:.4f}")
print(f"  F1 Macro: {f1_macro:.4f} | F1 Weighted: {f1_weighted:.4f}")
print(f"  Business Cost: {holdout_cost} (FN={fn}×{FN_COST} + FP={fp}×{FP_COST})")
print(f"  Per-class PR-AUC: {pr_aucs}")
print(f"\nClassification Report:")
print(classification_report(y_holdout, y_hold_pred, labels=list(range(n_classes)),
                            target_names=class_names))

results = {
    'model': model, 'le': le, 'accuracy': acc, 'f1_macro': f1_macro,
    'f1_weighted': f1_weighted, 'confusion_matrix': cm,
    'class_names': class_names, 'report': report,
    'feature_importances': model.feature_importances_,
    'pr_aucs': pr_aucs, 'holdout_cost': holdout_cost,
    'fn_cost': FN_COST, 'fp_cost': FP_COST,
    'holdout_fn': int(fn), 'holdout_fp': int(fp),
    'features_clean': features_clean,
    'best_iteration': model.best_iteration, 'best_params': bp,
}


Loaded saved Optuna params from optuna_best_params.json

Target: hot_part_fail
  Using saved params: {'max_depth': 5, 'learning_rate': 0.03261736917391224, 'n_estimators': 467, 'min_child_weight': 9, 'subsample': 0.9209788908984441, 'colsample_bytree': 0.5200705716811362, 'reg_alpha': 7.085782228975299e-06, 'reg_lambda': 0.00020044697585510238}
  Early stopping: iteration 466 / 467
  Val threshold: 0.59 (cost=4049, FN=388, FP=169)

  Holdout: Acc=0.9609 F1M=0.6886 PR-AUC=0.3655 Recall@0.8prec=0.2645
  Cost=4311 (FN=415×10 + FP=161×1)

Target: cold_part_fail
  Using saved params: {'max_depth': 5, 'learning_rate': 0.04888249007505776, 'n_estimators': 355, 'min_child_weight': 9, 'subsample': 0.7962043872542451, 'colsample_bytree': 0.32486104345078975, 'reg_alpha': 6.8173484547239325e-06, 'reg_lambda': 0.036396406169757534}
  Early stopping: iteration 354 / 355
  Val threshold: 0.65 (cost=4444, FN=432, FP=124)

  Holdout: Acc=0.9607 F1M=0.6923 PR-AUC=0.3668 Recall@0.8prec=0.2641
  Cost=47

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# --- Confusion Matrix ---
fig_cm = go.Figure(data=go.Heatmap(
    z=cm, x=class_names, y=class_names,
    colorscale='Blues', text=cm, texttemplate='%{text}',
))
fig_cm.update_layout(title='Confusion Matrix (Holdout)',
                     xaxis_title='Predicted', yaxis_title='Actual',
                     template='plotly_white', yaxis=dict(autorange='reversed'))
fig_cm.show()

# --- Feature Importance ---
imp = model.feature_importances_
top_idx = np.argsort(imp)[-20:]
fi_df = pd.DataFrame({
    'Feature': [features_clean[i] for i in top_idx],
    'Importance': imp[top_idx],
}).sort_values('Importance')
fig_fi = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
                color='Importance', color_continuous_scale='Viridis',
                title='Top 20 Feature Importances')
fig_fi.update_layout(template='plotly_white', height=500, showlegend=False)
fig_fi.show()

# --- Summary ---
print(f"\nModel: XGBoost multi-class ({n_classes} classes: {class_names})")
print(f"Holdout cost: {holdout_cost} | Accuracy: {acc:.4f} | F1 Macro: {f1_macro:.4f}")
print(f"Per-class PR-AUC: {pr_aucs}")